# 01 — Text Generation with Autoregressive (GPT-style) Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand the **autoregressive principle** behind GPT models: predict the next token, then feed it back in
- **Build and train a character-level language model on a real text corpus** that generates text with exactly that mechanism
- Control generation with **temperature** and seed text
- Know how this scales up to GPT-2/GPT-4 (and why we do not download one here)

## 🔗 Where this fits

**Builds on:** Course 07 (AIAT 121) — Unit 4, lesson 05 (GPT-2 as a black box) and Course 08 (AIAT 122) — Unit 3 (RNN/LSTM training): here you train the language model yourself, still by loss + gradient descent.

**Used later in:** Course 10 — Unit 2, lessons 02-06, all of which reuse this character-level model.

## 📊 Data used in this notebook

Two **real** text corpora, no invented sentences:
1. **20 Newsgroups** (`sklearn.datasets.fetch_20newsgroups`) — ~20k Usenet posts written by real people in the early 1990s. We use the `sci.space` group.
2. **Montgomery County 911 emergency calls** (`Course 04/datasets/raw/montgomery_911_calls.csv`) — real dispatch records; we generate from the free-text `title` field.

Real text matters here: a hand-typed sentence repeated ten times can be *memorised* in a few hundred steps, which makes a language model look far better than it is. On real prose the model has to learn genuine character statistics, and you see honestly how far a small model gets.

---

This notebook covers practical activities from **Course 10, Unit 2**:
- Implementing text generation with GPT-style autoregressive models

---

## Introduction

**GPT models** are autoregressive language models: they model p(next token | all previous tokens) and generate by sampling one token at a time. This notebook demonstrates the same principle with a character-level LSTM trained on real text — identical generation mechanics, no multi-GB download. Fine-tuning pretrained models is the subject of example 02; quality metrics are example 06.


## How Autoregressive Generation Works

Training: slice text into (context → next character) pairs and minimize cross-entropy on the next character. Generation: start from a seed, predict a distribution over the next character, **sample** from it (temperature scales the logits first: low = safe/repetitive, high = diverse/risky), append, repeat. Every LLM you have used works this way, token by token.


In [1]:
# WHAT/WHY: name the real GPT models this lesson maps onto, and state exactly
# what our stand-in shares with them (the autoregressive mechanism).
import torch, torch.nn as nn, torch.optim as optim, numpy as np
print(f'PyTorch {torch.__version__}')
print('Text Generation with Autoregressive Language Models')
print('=' * 60)
print('GPT models (concept):')
print('  - GPT-2: 117M-1.5B parameter transformer trained on web text')
print('  - Core idea: predict next token, left-to-right (causal LM)')
print()
print('This notebook demonstrates the SAME autoregressive principle')
print('using a character-level LSTM - identical inference mechanics,')
print('no multi-GB download required.')


PyTorch 2.13.0
Text Generation with Autoregressive Language Models
GPT models (concept):
  - GPT-2: 117M-1.5B parameter transformer trained on web text
  - Core idea: predict next token, left-to-right (causal LM)

This notebook demonstrates the SAME autoregressive principle
using a character-level LSTM - identical inference mechanics,
no multi-GB download required.


In [2]:
# WHAT/WHY: build the full pipeline on a REAL corpus — load genuine Usenet
# posts, encode them as characters, train next-character prediction, then
# generate by sampling in a loop. Real text (not a repeated toy sentence) is
# what forces the model to learn actual English statistics instead of
# memorising three lines.
import re
from sklearn.datasets import fetch_20newsgroups

torch.manual_seed(42); np.random.seed(42)

# ── REAL DATA: Usenet posts from the sci.space newsgroup ──────────────────
# remove=(headers, footers, quotes) strips the mail headers and quoted replies
# so we are left with prose the poster actually wrote.
news = fetch_20newsgroups(subset='train', categories=['sci.space'],
                          remove=('headers', 'footers', 'quotes'))
raw = " ".join(news.data)

# Light normalisation: lowercase and keep only letters, digits, space, . and ,
# — this shrinks the character vocabulary so a small LSTM can learn it on CPU.
corpus = re.sub(r'[^a-z0-9 .,]', ' ', raw.lower())
corpus = re.sub(r'\s+', ' ', corpus).strip()[:20000]
print(f'Real corpus: {len(news.data)} sci.space posts → first {len(corpus):,} characters used')
print(f'Sample: {corpus[:160]!r}')

# ── Vocabulary: map each character to an integer id (a tiny "tokenizer") ──
chars = sorted(set(corpus))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
V, S = len(chars), 30
print(f'Vocabulary: {V} characters | context window: {S} characters')

# ── Dataset: every 30-char window predicts the character that follows it ──
X = torch.tensor([[c2i[corpus[j+k]] for k in range(S)] for j in range(len(corpus)-S)], dtype=torch.long)
y = torch.tensor([c2i[corpus[j+S]] for j in range(len(corpus)-S)], dtype=torch.long)
print(f'Training windows: {X.shape[0]:,}')

# ── Model: embedding → LSTM → linear scores over the vocabulary ───────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(V, 32)
        self.lstm = nn.LSTM(32, 128, batch_first=True)
        self.fc   = nn.Linear(128, V)
    def forward(self, x):
        return self.fc(self.lstm(self.emb(x))[0][:, -1, :])

model = CharLM()
opt   = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(X, y), batch_size=256, shuffle=True)

# ── Training: minimize next-character cross-entropy ───────────────────────
for epoch in range(15):
    model.train(); el = 0
    for xb, yb in loader:
        opt.zero_grad(); loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step(); el += loss.item()
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}: loss={el/len(loader):.4f}')

# ── Sampling with temperature (like GPT decoding) ─────────────────────────
def generate(seed, n=120, temperature=0.8):
    model.eval(); out = seed
    ctx = [c2i.get(c, 0) for c in seed[-S:]]
    for _ in range(n):
        x = torch.tensor([ctx], dtype=torch.long)
        with torch.no_grad():
            logits = model(x)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        nxt = int(np.random.choice(V, p=probs))
        out += i2c[nxt]; ctx = ctx[1:] + [nxt]
    return out

np.random.seed(0)
print('\nSeed → generated (real-text model):')
print(repr(generate('the space shuttle ', n=120, temperature=0.8)))
print(repr(generate('nasa launched the ', n=120, temperature=1.0)))
print('\nRead this honestly: 20k characters and a 128-unit LSTM buy you English')
print('letter and short-word statistics, not meaning. Scaling the SAME')
print('autoregressive recipe to billions of tokens is what produces GPT-4.')


Real corpus: 593 sci.space posts → first 20,000 characters used
Sample: 'any lunar satellite needs fuel to do regular orbit corrections, and when its fuel runs out it will crash within months. the orbits of the apollo motherships cha'
Vocabulary: 39 characters | context window: 30 characters
Training windows: 19,970


Epoch 5: loss=2.0538


Epoch 10: loss=1.7122


Epoch 15: loss=1.5006

Seed → generated (real-text model):
'the space shuttle on spectors stabe tope the took on spestant stacing was performed the wething firical semons tech saunnt. statsion. the '
'nasa launched the sotell miked in spections, usy terectionce tro ewal a spane consument in with acouncading was 40 lbo the will conale. go'

Read this honestly: 20k characters and a 128-unit LSTM buy you English
letter and short-word statistics, not meaning. Scaling the SAME
autoregressive recipe to billions of tokens is what produces GPT-4.


## 🌍 Real-World Worked Example — Generating Emergency-Dispatch Text

**Industry context:**
- GitHub Copilot generates code token by token using a GPT-style model
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

Here we point the same character-level model at a **second real corpus with a very different style**: the free-text `title` field of the **Montgomery County (PA) 911 call log** — ~123 MB of genuine dispatch records. Emergency-dispatch text is short, capitalised, highly templated (`EMS: BACK PAINS/INJURY`), so the model should learn its structure faster than it learned free-form Usenet prose. Comparing the two runs shows you something real: **how much a language model learns depends on how repetitive its training corpus is.**


In [3]:
# WHAT/WHY: the same pipeline on a SECOND real corpus — 911 dispatch call
# titles — with a 2-layer LSTM. Watch the loss fall much further than it did
# on Usenet prose: this corpus is far more templated, so it is easier to model.
import pandas as pd

torch.manual_seed(42); np.random.seed(42)

# ── REAL DATA: Montgomery County 911 call log ─────────────────────────────
# Relative path: the dataset lives once in the repo and is shared by courses.
# usecols + nrows keep a 123 MB file classroom-sized and fast to load.
CALLS = '../../../Course 04/datasets/raw/montgomery_911_calls.csv'
calls = pd.read_csv(CALLS, usecols=['title'], nrows=20000)
print(f'Loaded {len(calls):,} real 911 call records')
print('Most common call types:')
print(calls['title'].value_counts().head(5).to_string())

# One long stream of dispatch titles, separated by " | " so the model can learn
# where one call ends and the next begins.
text = ' | '.join(calls['title'].astype(str).tolist())[:20000]
print(f'\nCorpus: {len(text):,} characters')
print(f'Sample: {text[:150]!r}')

# ── Vocabulary: map each character to an integer id ───────────────────────
chars  = sorted(set(text))
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for c, i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]
print(f'Vocabulary: {VOCAB} characters')

# ── Dataset: every 20-char window predicts the next character ─────────────
SEQ_LEN = 20
X_t = torch.tensor([enc[i:i+SEQ_LEN] for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)
y_t = torch.tensor([enc[i+SEQ_LEN]   for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM2(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model2  = CharLM2()
opt     = optim.Adam(model2.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

# ── Training: 600 steps, a fresh random 256-window batch each time ────────
for epoch in range(600):
    model2.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model2(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 150 == 0:
        print(f"Step {epoch} — loss: {loss.item():.3f}")
print(f"Step 599 — loss: {loss.item():.3f}")

# ── Text Generation (temperature sampling) ────────────────────────────────
def generate2(seed_str, steps=120, temperature=0.8):
    model2.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model2(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = int(np.random.choice(len(probs), p=probs))
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

np.random.seed(0)
print("\n── Generated dispatch titles ───────────────────────────────────")
print(generate2("EMS: ", steps=120))
print()
print(generate2("Fire: ", steps=120))
print("\nSame mechanism as ChatGPT — one token at a time — but note the")
print("difference from the Usenet run above: a templated corpus is learned")
print("much more sharply than free prose. Corpus statistics, not architecture,")
print("explain most of the gap.")


Loaded 20,000 real 911 call records
Most common call types:
title
Traffic: VEHICLE ACCIDENT -    4798
Traffic: DISABLED VEHICLE -    2077
Fire: FIRE ALARM               1059
EMS: RESPIRATORY EMERGENCY     1058
EMS: FALL VICTIM               1010

Corpus: 20,000 characters
Sample: 'EMS: BACK PAINS/INJURY | EMS: DIABETIC EMERGENCY | Fire: GAS-ODOR/LEAK | EMS: CARDIAC EMERGENCY | EMS: DIZZINESS | EMS: HEAD INJURY | EMS: NAUSEA/VOMI'
Vocabulary: 36 characters
Step 0 — loss: 3.573


Step 150 — loss: 0.556


Step 300 — loss: 0.183


Step 450 — loss: 0.244


Step 599 — loss: 0.190

── Generated dispatch titles ───────────────────────────────────
EMS: UNRESPONSIVE SUBJECT | Traffic: VEHICLE ACCIDENT - | EMS: HEAD INJURY | Traffic: VEHICLE ACCIDENT - | Traffic: VEHICLE A

Fire: VEHICLE - | EMS: SUBJECT IN PIAT ON T ALT AUST ARM | EMS: UNKNOWN MEDICAL EMERGENCY | EMS: CARDIAC EMERGENCY | EMS: HEAD

Same mechanism as ChatGPT — one token at a time — but note the
difference from the Usenet run above: a templated corpus is learned
much more sharply than free prose. Corpus statistics, not architecture,
explain most of the gap.


## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

You built **autoregressive text generation** — the core mechanism behind GPT, ChatGPT, and LLaMA — and trained it twice on **real** text: 593 free-form Usenet posts from `sci.space`, and 20,000 templated 911 dispatch titles.

Three things the real data taught you that a toy corpus could not:
1. The model learns to predict the next token given context; generation is that prediction, fed back in.
2. **Temperature** trades coherence for diversity at decoding time, on a fixed model.
3. **Corpus statistics dominate.** The identical recipe reached a final next-character loss of **1.50 on free prose** but **0.19 on the dispatch log** — and it shows: the prose model emits English-shaped nonsense (`the space shuttle on spectors stabe tope...`), while the dispatch model reproduces genuine call types (`EMS: UNRESPONSIVE SUBJECT | Traffic: VEHICLE ACCIDENT -`). When a language model looks impressive, always ask how predictable its training text was.

Scaling this architecture — and this data appetite — to billions of parameters and trillions of tokens is what creates foundation models.
